# Derm-Vision: ISIC 2019 Skin Lesion Classification — Training

This notebook runs the full training pipeline on Google Colab with GPU.

**Setup**: Upload the ISIC 2019 dataset to your Google Drive at:
`My Drive/derm-vision-data/ISIC_2019_Training_Input/` (images + CSVs)

## 1. GPU Check & Setup

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/duylam1407/derm-vision.git /content/derm-vision
%cd /content/derm-vision

In [ ]:
!pip install -q timm albumentations wandb grad-cam

## 3. Link Data from Google Drive

Create symlinks so the project can find the data without copying ~10GB.

In [ ]:
import os

# === CONFIGURE THIS ===
# Path to your ISIC data folder in Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/derm-vision-data"
# ======================

# Symlink data into the project
os.makedirs("data/raw", exist_ok=True)

files_to_link = [
    ("ISIC_2019_Training_Input", "data/raw/ISIC_2019_Training_Input"),
    ("ISIC_2019_Training_GroundTruth.csv", "data/raw/ISIC_2019_Training_GroundTruth.csv"),
    ("ISIC_2019_Training_Metadata.csv", "data/raw/ISIC_2019_Training_Metadata.csv"),
]

for src_name, dst_path in files_to_link:
    src = os.path.join(DRIVE_DATA_DIR, src_name)
    if os.path.exists(dst_path):
        os.remove(dst_path)
    os.symlink(src, dst_path)
    print(f"{dst_path} -> {src} ({'OK' if os.path.exists(src) else 'MISSING!'})")

## 4. Create Data Splits (if not already committed)

In [ ]:
if not os.path.exists("data/splits/train.csv"):
    !python scripts/create_splits.py
else:
    print("Splits already exist, skipping.")

## 5. Update Config for Colab

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# Point to the correct image directory
# Adjust this if your Drive folder structure is different
cfg["data"]["data_dir"] = "data/raw/ISIC_2019_Training_Input/ISIC_2019_Training_Input"

# Save checkpoints to Drive so they persist
drive_output = "/content/drive/MyDrive/derm-vision-data/outputs"
os.makedirs(os.path.join(drive_output, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(drive_output, "results"), exist_ok=True)
cfg["output"]["checkpoint_dir"] = os.path.join(drive_output, "checkpoints")
cfg["output"]["results_dir"] = os.path.join(drive_output, "results")

with open("configs/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Config updated.")
print(f"  data_dir: {cfg['data']['data_dir']}")
print(f"  checkpoint_dir: {cfg['output']['checkpoint_dir']}")

## 6. Login to W&B

In [ ]:
import wandb
wandb.login()

## 7. Train

In [ ]:
!python -m src.train --config configs/config.yaml

## 8. Evaluate on Test Set

In [ ]:
from src.dataset import ISICDataset, CLASS_NAMES
from src.transforms import get_val_transforms
from src.models.efficientnet import EfficientNetB3Classifier
from src.evaluate import compute_metrics, print_classification_report, plot_confusion_matrix
from torch.utils.data import DataLoader

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model
model = EfficientNetB3Classifier(
    num_classes=cfg["data"]["num_classes"],
    pretrained=False,
    dropout=cfg["model"]["dropout"],
).to(device)
ckpt_path = os.path.join(cfg["output"]["checkpoint_dir"], "best_model.pth")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

# Test dataset
test_dataset = ISICDataset(
    image_dir=cfg["data"]["data_dir"],
    labels_csv=os.path.join(cfg["data"]["splits_dir"], "test.csv"),
    transform=get_val_transforms(cfg["data"]["image_size"]),
)
test_loader = DataLoader(test_dataset, batch_size=cfg["training"]["batch_size"], num_workers=2)

# Run inference
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.tolist())

# Metrics
metrics = compute_metrics(all_labels, all_preds)
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
print(f"Weighted F1:       {metrics['weighted_f1']:.4f}")
print()
print(print_classification_report(all_labels, all_preds, CLASS_NAMES))

# Confusion matrix
save_path = os.path.join(cfg["output"]["results_dir"], "confusion_matrix.png")
plot_confusion_matrix(all_labels, all_preds, CLASS_NAMES, save_path=save_path)